# 03 — Executive Dashboard Notebook

**Purpose:** Pre-built visualizations ready for Streamlit deployment.
**Dataset:** 130,038 real Amazon Electronics reviews (5-core subset)
**Source:** http://jmcauley.ucsd.edu/data/amazon/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load real data
df = pd.read_csv('../data/amazon_reviews_electronics_5core.csv')
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['reviewYear'] = df['reviewDate'].dt.year
df['reviewMonth'] = df['reviewDate'].dt.month
df['reviewYearMonth'] = df['reviewDate'].dt.to_period('M').astype(str)
df['reviewLength'] = df['reviewText'].fillna('').str.len()
df['helpfulnessRatio'] = np.where(df['helpful_total'] > 0, df['helpful_upvotes'] / df['helpful_total'], np.nan)

print(f'Records: {len(df):,}')
print(f'Date range: {df.reviewDate.min().date()} to {df.reviewDate.max().date()}')
print(f'Products: {df.asin.nunique():,} | Reviewers: {df.reviewerID.nunique():,}')

## Section A — Product Performance Leaderboard

In [ ]:
# Top products by volume + rating quality
product_stats = df.groupby('asin').agg(
    review_volume=('overall', 'size'),
    avg_rating=('overall', 'mean'),
    helpful_total=('helpful_total', 'sum'),
    helpful_upvotes=('helpful_upvotes', 'sum')
).reset_index()

product_stats['helpfulness_rate'] = np.where(product_stats['helpful_total'] > 0,
                                              product_stats['helpful_upvotes'] / product_stats['helpful_total'], 0)
product_stats['engagement_score'] = (
    product_stats['avg_rating'] * 0.4 +
    np.log1p(product_stats['review_volume']) * 0.4 +
    product_stats['helpfulness_rate'] * 0.2
)

# Filter for statistical relevance
qualified = product_stats[product_stats['review_volume'] >= 10].copy()
qualified = qualified.sort_values('engagement_score', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(qualified))
colors = plt.cm.RdYlGn(qualified['avg_rating'] / 5)

ax.barh(y_pos, qualified['engagement_score'], color=colors, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{a[:10]}... ({int(v)} reviews)" for a, v in zip(qualified['asin'], qualified['review_volume'])], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Engagement Score')
ax.set_title('Top 20 Electronics Products by Engagement Score\n(Color = avg rating intensity)')
plt.tight_layout()
plt.show()

qualified[['asin', 'review_volume', 'avg_rating', 'helpfulness_rate', 'engagement_score']].head(10)

## Section B — Trend Alerts: Rating Degradation Detection

In [ ]:
# YoY rating change for products with sufficient data
yearly_product = df.groupby(['asin', 'reviewYear'])['overall'].agg(['mean', 'count']).reset_index()
yearly_product = yearly_product[yearly_product['count'] >= 5]

# Pivot to get consecutive year comparisons
pivot = yearly_product.pivot(index='asin', columns='reviewYear', values='mean')

# Calculate 2013 vs 2012 degradation for products present in both
if 2012 in pivot.columns and 2013 in pivot.columns:
    degrade = pivot[[2012, 2013]].dropna()
    degrade['change'] = degrade[2013] - degrade[2012]
    degrade['abs_change'] = degrade['change'].abs()
    degrade = degrade.sort_values('abs_change', ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['red' if c < 0 else 'green' for c in degrade['change'].head(30)]
    ax.bar(range(30), degrade['change'].head(30), color=colors, edgecolor='black')
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('Product Rank (by magnitude of change)')
    ax.set_ylabel('Rating Change (2013 vs 2012)')
    ax.set_title('Top 30 Products with Largest YoY Rating Swing (2012→2013)')
    plt.tight_layout()
    plt.show()
    
    print(f"Products with sharp decline (-0.5+): {(degrade['change'] <= -0.5).sum()}")
    print(f"Products with strong improvement (+0.5+): {(degrade['change'] >= 0.5).sum()}")
else:
    print('Insufficient consecutive year data for degradation analysis')

## Section C — Sentiment Tracker: Monthly Rating + Volume

In [ ]:
monthly = df.groupby('reviewYearMonth').agg(
    volume=('overall', 'size'),
    avg_rating=('overall', 'mean'),
    std_rating=('overall', 'std')
).reset_index()
monthly['reviewYearMonth'] = pd.to_datetime(monthly['reviewYearMonth'])
monthly['rating_3mo'] = monthly['avg_rating'].rolling(3, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Volume
axes[0].plot(monthly['reviewYearMonth'], monthly['volume'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly['reviewYearMonth'], monthly['volume'], alpha=0.3, color='steelblue')
axes[0].set_ylabel('Reviews / Month')
axes[0].set_title('Monthly Review Volume (Electronics)')

# Rating with 3-month smooth
axes[1].plot(monthly['reviewYearMonth'], monthly['avg_rating'], color='lightgray', alpha=0.5, label='Monthly')
axes[1].plot(monthly['reviewYearMonth'], monthly['rating_3mo'], color='darkgreen', linewidth=2, label='3-month avg')
axes[1].axhline(df['overall'].mean(), color='red', linestyle='--', label=f'Overall mean: {df["overall"].mean():.2f}')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('Monthly Average Rating Trend')
axes[1].legend()
axes[1].set_ylim(3.5, 5.0)

plt.tight_layout()
plt.show()

# Alert: months below 3.8 rating
alerts = monthly[monthly['avg_rating'] < 3.8]
print(f"Months with avg rating below 3.8: {len(alerts)}")
if len(alerts) > 0:
    print(alerts[['reviewYearMonth', 'avg_rating', 'volume']].to_string(index=False))

## Section D — Seasonal Heatmap

In [ ]:
# Prepare seasonal data
seasonal = df.groupby(['reviewYear', 'reviewMonth']).size().reset_index(name='volume')
seasonal_pivot = seasonal.pivot(index='reviewYear', columns='reviewMonth', values='volume').fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(seasonal_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Review Volume'})
ax.set_xlabel('Month')
ax.set_ylabel('Year')
ax.set_title('Seasonal Review Volume Heatmap (Electronics)')
plt.tight_layout()
plt.show()